In [1]:
import sys
from pathlib import Path
print("Python:", sys.version)
print("CWD:", Path.cwd())
# ensure repo top is importable
sys.path.insert(0, str(Path.cwd()))  # assume running from /SIR-25-26/referee
print("sys.path[0]:", sys.path[0])

Python: 3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]
CWD: /home/jupyter-ashah2/SIR-25-26/referee
sys.path[0]: /home/jupyter-ashah2/SIR-25-26/referee


In [2]:
# Ensure working directory is the referee repo root
import os
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
print("cwd:", REPO_ROOT)
expected = Path.home() / "SIR-25-26" / "referee"
if REPO_ROOT != expected:
    print("Warning: current working dir is not expected repo root.")
    try:
        os.chdir(expected)
        print("Changed cwd to:", Path.cwd())
    except Exception as e:
        print("Could not chdir to expected path:", expected, "error:", e)

# Make sure the repo root is on sys.path so `import dataset` works
sys.path.insert(0, str(Path.cwd()))
print("sys.path[0]:", sys.path[0])

# Now import the dataloaders from the dataset package
try:
    from dataset.dataloader import TrainDataset, TestDataset
    print("Imported dataset.dataloader successfully.")
except Exception as e:
    print("Import failed:", e)
    raise

cwd: /home/jupyter-ashah2/SIR-25-26/referee
sys.path[0]: /home/jupyter-ashah2/SIR-25-26/referee
Imported dataset.dataloader successfully.


In [3]:
!pip show av
import av

print("Making TrainDataset...")
train_ds = TrainDataset(json_path="data/train_targets_clean.json", transform_fn=None, ref_csv_path="data/all_real_with_split_fixed.csv")
print("Train size:", len(train_ds))

sample = train_ds[0]
print("\nSample keys:", list(sample.keys()))
print("target_video type:", type(sample["target_video"]), "shape:", getattr(sample["target_video"], "shape", None))
print("target_audio type:", type(sample["target_audio"]), "shape:", getattr(sample["target_audio"], "shape", None))
print("reference_video shape:", getattr(sample["reference_video"], "shape", None))
print("reference_audio shape:", getattr(sample["reference_audio"], "shape", None))
print("fake_label:", sample["fake_label"])
print("id_label:", sample["id_label"])

print("\nNow TestDataset (pairs):")
test_ds = TestDataset("data/test_pairs_fixed.json")
print("Test size:", len(test_ds))
s2 = test_ds[0]
print("Test sample keys:", list(s2.keys()))

Name: av
Version: 16.1.0
Summary: Pythonic bindings for FFmpeg's libraries.
Home-page: https://pyav.basswood-io.com
Author: 
Author-email: WyattBlue <wyattblue@auto-editor.com>, Jeremy Lainé <jeremy.laine@m4x.org>
License-Expression: BSD-3-Clause
Location: /home/jupyter-ashah2/.local/lib/python3.10/site-packages
Requires: 
Required-by: 
Making TrainDataset...
Train size: 14003


/home/jupyter-ashah2/.local/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(



Sample keys: ['target_video', 'target_audio', 'reference_video', 'reference_audio', 'fake_label', 'id_label']
target_video type: <class 'torch.Tensor'> shape: torch.Size([367, 3, 224, 224])
target_audio type: <class 'torch.Tensor'> shape: torch.Size([235520])
reference_video shape: torch.Size([367, 3, 224, 224])
reference_audio shape: torch.Size([235520])
fake_label: tensor(1)
id_label: tensor(0)

Now TestDataset (pairs):
Test size: 6464
Test sample keys: ['target_video', 'target_audio', 'reference_video', 'reference_audio', 'fake_label', 'id_label']


In [4]:
# DIAGNOSTICS
from pathlib import Path
import torch, json
from dataset.dataloader import TrainDataset
from dataset.dataset_utils import get_video_and_audio

ds = TrainDataset(json_path="data/train_targets.json", transform_fn=None, ref_csv_path="data/all_real_with_split_fixed.csv")
print("train size", len(ds))

def check_samples(ds, N=200):
    bad = []
    for idx in range(min(N, len(ds))):
        item = ds.samples[idx]             # underlying json entry
        tgt_path = item["file_path"]
        tgt_id = item["id"]
        split = item.get("split", "train")
        # pick a reference that TrainDataset would use
        refs = ds.reference_map.get((tgt_id, split), [])
        if not refs:
            bad.append((idx, "NO_REFS_IN_CSV", tgt_path, tgt_id, split))
            print("bad")
            continue
        ref_path = refs[0]
        try:
            tv, ta, tmeta = get_video_and_audio(tgt_path, get_meta=True)
        except Exception as e:
            bad.append((idx, "TARGET_READ_ERROR", tgt_path, ref_path, str(e)))
            print("bad")
            continue
        try:
            rv, ra, rmeta = get_video_and_audio(ref_path, get_meta=True)
        except Exception as e:
            bad.append((idx, "REF_READ_ERROR", tgt_path, ref_path, str(e)))
            print("bad")
            continue

        # detect empty tensors
        if (isinstance(rv, torch.Tensor) and rv.numel()==0) or (isinstance(ra, torch.Tensor) and ra.numel()==0):
            bad.append((idx, "REF_EMPTY", tgt_path, ref_path, tv.shape, ta.shape, getattr(rv,"shape",None), getattr(ra,"shape",None)))
    return bad

bad = check_samples(ds, N=200)
print("Bad entries (first 200):", len(bad))
for b in bad[:20]:
    print(b)


train size 14003


/home/jupyter-ashah2/.local/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


Bad entries (first 200): 0


In [2]:
import av, sys, inspect
print("av module file:", getattr(av, "__file__", None))
print("av package path:", av.__path__ if hasattr(av, "__path__") else None)
print("av attrs contains AVError?", "AVError" in dir(av))
# show top-level members (helpful)
print("top-level members:", sorted([n for n in dir(av) if not n.startswith("_")])[:60])
# show first 6 entries of sys.path
print("sys.path (first 6):")
for p in sys.path[:6]:
    print(" ", p)

av module file: /home/jupyter-ashah2/.local/lib/python3.10/site-packages/av/__init__.py
av package path: ['/home/jupyter-ashah2/.local/lib/python3.10/site-packages/av']
av attrs contains AVError? False
top-level members: ['AudioCodecContext', 'AudioFifo', 'AudioFormat', 'AudioFrame', 'AudioLayout', 'AudioResampler', 'AudioStream', 'BSFNotFoundError', 'BitStreamFilterContext', 'BlockingIOError', 'BrokenPipeError', 'BufferTooSmallError', 'BugError', 'ChildProcessError', 'Codec', 'CodecContext', 'ConnectionAbortedError', 'ConnectionRefusedError', 'ConnectionResetError', 'ContainerFormat', 'DecoderNotFoundError', 'DemuxerNotFoundError', 'EOFError', 'EncoderNotFoundError', 'ErrorType', 'ExitError', 'ExperimentalError', 'ExternalError', 'FFmpegError', 'FileExistsError', 'FileNotFoundError', 'FilterNotFoundError', 'HTTPBadRequestError', 'HTTPClientError', 'HTTPError', 'HTTPForbiddenError', 'HTTPNotFoundError', 'HTTPOtherClientError', 'HTTPServerError', 'HTTPUnauthorizedError', 'HWConfig', 'In

In [3]:
import torch, sys
print("python exe:", sys.executable)
print("torch version:", getattr(torch, "__version__", None))
print("torch.cuda.is_available():", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, "->", torch.cuda.get_device_name(i))

python exe: /home/jupyter-ashah2/.conda/envs/referee/bin/python
torch version: 2.5.1
torch.cuda.is_available(): True
cuda device count: 2
0 -> Quadro RTX 8000
1 -> Quadro RTX 8000
